## Upload to MotherDuck notebook (OPTIONAL)

This Jupyter Notebook is designed to help you upload a local DuckDB database (file) to the MotherDuck server. 

This allows you to share and collaborate on your database with others easily.

### Purpose

The main purpose of this notebook is to:

1. Set up logging: Customise logging to track the process and any issues.
2. Load secrets: Load necessary secrets (like tokens/password) from a file to authenticate with MotherDuck.
3. Check local database: Verify if a local DuckDB database exists.

Upload database: Copy the local DuckDB database to the MotherDuck server.
Interact with the database: Perform SQL queries on the uploaded database.

### Steps

- Import necessary libraries: The notebook imports various libraries required for file handling, database operations, and logging.
- Customise Logger: Sets up a logger to track the process and any issues.
- Load Secrets: Loads the MotherDuck token from a `.secrets.toml` file.
- Check Local Database: Checks if the local DuckDB database exists.

Upload Database: Copies the local DuckDB database to the MotherDuck server.
Interact with the Database: Connects to the uploaded database and performs SQL queries.

### How to Use

Ensure Dependencies: Make sure you have all the required libraries installed. You can install them using pip:
Bash
loguru
Prepare Secrets File: Create a .secrets.toml file in the parent directory of your current working directory. This file should contain your MotherDuck token:
"

Run the Notebook: Open the notebook in Jupyter and run each cell sequentially. The notebook will:
- Set up logging.
- Load the MotherDuck token.
- Check for the local DuckDB database.
- Upload the database to MotherDuck.
- Allow you to interact with the uploaded database.

Collaborate: Once the database is uploaded to MotherDuck, you can share access with your collaborators. They can connect to the database using sharing.

#### Example Output

- Database Upload: The notebook will log the creation of the database on MotherDuck.
- SQL Queries: You can run SQL queries on the uploaded database and see the results directly in the notebook.

#### Troubleshooting

- Logging: Check the log file (`DuckDBManager.log`) for any errors or issues during the process.
- Secrets: Ensure your `.secrets.toml` file is correctly formatted and contains the right token.

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import duckdb
import toml
from loguru import logger

In [ ]:
def stdout_filter(record):
    return record["level"].name == "INFO" and "ECHO" in record["message"]

def customise_logger(logger_filename="DuckDBManager.log", logger_size=10, filter_func=stdout_filter, level="INFO"):
    try:
        logger.remove(0) # Remove default output to sys.stderr
        logger.add(sys.stdout, filter=filter_func, level=level)     # Add sys.stdout logging of select messages
        logger.add(logger_filename, rotation=f"{logger_size} MB", level=level)  # logs will be written to file.log
        logger.info(f"ECHO: Logging to {logger_filename}: size = {logger_size} MB")
        logger.info("ECHO: Limited logging to stdout using `stdout_filter`")
    except Exception as e:
        logger.error(f"Error with customising logger: {e}")

In [ ]:
customise_logger()

In [ ]:
logger.info(f"ECHO - Local DuckDB version: {duckdb.__version__}")

In [ ]:
SECRETS_TOML = Path.cwd().parent / ".secrets.toml"

In [ ]:
def set_md_token(secrets_file):
    """
    Load secrets from a file and set the motherduck_token environment variable.
    """
    if "motherduck_token" not in os.environ:
        try:
            secrets = toml.load(secrets_file)
            os.environ["motherduck_token"] = secrets["MD_TOKEN"]
            logger.info(f"ECHO: Secrets loaded from {secrets_file} and MotherDuck environment variable set")
        except Exception as e:
            logger.error(f"Failed to load secrets: {e}")
            raise
    else:
        logger.info("ECHO: MD environment variable already set.")

In [ ]:
def check_local_duckdb_database():
    """
    Checks if a local DuckDB database exists. It returns the path to the database if it exists, and None otherwise.
    """
    DATA_DIR = Path("../data")
    EXPERIMENT_TYPE = "RPE"  # or CB
    assert DATA_DIR.exists()
    local_ddb = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023_Q4.ddb"  # DuckDB database name

    logger.info(f"ECHO - Database file: {local_ddb.resolve()}")

    if local_ddb.exists():
        return local_ddb
    else:
        return None

In [ ]:
local_ddb = check_local_duckdb_database()

In [ ]:
def copy_local_ddb_to_md(local_ddb, add_date_suffix=True):
    set_md_token(SECRETS_TOML)
    try:
        if add_date_suffix:
            date_suffix = datetime.now().strftime("_%d_%B_%Y")
        else:
            date_suffix = ""
        remote_db_name = f"{local_ddb.stem.lower()}{date_suffix}"
        con_md = duckdb.connect("md:")
        con_md.sql("USE default_db;")    # To avoid trying to drop database if it is the default in MD
        con_md.sql(f"DROP DATABASE IF EXISTS {remote_db_name};")
        con_md.sql(f"CREATE OR REPLACE DATABASE {remote_db_name} FROM '{str(local_ddb)}';")
        logger.info(f"Created database {remote_db_name} on https://motherduck.com")
        return remote_db_name, con_md
    except Exception as e:
        logger.error(f"Failed to copy local database to MotherDuck server: {e}")
        raise


In [ ]:
remote_db_name, con_md = copy_local_ddb_to_md(local_ddb, add_date_suffix=True)

In [ ]:
con_md.sql("PRAGMA database_list;")

In [ ]:
remote_db_name

In [ ]:
con = duckdb.connect("md:")

In [ ]:
con.sql(f"USE '{remote_db_name}'")

In [ ]:
con.sql("SELECT * FROM Experiments")